<a href="https://colab.research.google.com/github/saurabhsen848/Generative-AI_Assignment/blob/main/Assignment_2_Generative_and_Discriminative_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/saurabhsen848/Generative-AI_Assignment.git

In [3]:
!pip install transformers -q

In [4]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import pandas as pd

In [5]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
def generate_text(prompt, max_length=60, **kwargs):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    output = model.generate(
        input_ids,
        max_length=max_length,
        pad_token_id=tokenizer.eos_token_id,
        **kwargs
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [7]:
prompt = "The scientist opened the door and found"

results = []

In [8]:
# Experiment 1: Low temperature -> more predictable, safe text
out1 = generate_text(prompt, do_sample=True, temperature=0.3, top_k=0, top_p=1.0)
results.append(("Temperature", 0.3, out1))

In [9]:
# Experiment 2: High temperature -> more random / less coherent text
out2 = generate_text(prompt, do_sample=True, temperature=1.5, top_k=0, top_p=1.0)
results.append(("Temperature", 1.5, out2))

In [16]:
# Experiment 3: Low top-k -> restricts choices to top 5 likely words
out3 = generate_text(prompt, do_sample=True, temperature=1.0, top_k=5, top_p=1.0)
results.append(("Top-k", 5, out3))

# --- Explanation for the error in the last cell (mYiYVUHvoUVP) ---
# The ValueError: "Length of values (5) does not match length of index (6)"
# is occurring because the 'observations' list in cell mYiYVUHvoUVP has 5 elements,
# but the 'results' list (which is used to create the DataFrame) has 6 elements.
#
# This discrepancy likely happened because the 'Repetition Penalty' experiment
# was added twice to the 'results' list, leading to an extra entry.
#
# To resolve this, you need to update the 'observations' list in cell mYiYVUHvoUVP
# to have 6 elements, matching the length of the 'results' list.
# For example, you could add a sixth observation string like:
# "Further observation for high repetition penalty".
# This will align the lengths and allow the DataFrame to be created without error.

In [11]:
# Experiment 4: Low top-p -> nucleus sampling
out4 = generate_text(prompt, do_sample=True, temperature=1.0, top_k=0, top_p=0.5)
results.append(("Top-p", 0.5, out4))

In [13]:
# Experiment 5: High repetition penalty -> discourages repeating words
out5 = generate_text(prompt, do_sample=True, temperature=1.0, top_k=0, top_p=1.0, repetition_penalty=2.0)
results.append(("Repetition Penalty", 2.0, out5))

In [14]:
for param, val, text in results:
    print(f"\n--- {param} = {val} ---")
    print(text)


--- Temperature = 0.3 ---
The scientist opened the door and found the body of a man who had been shot in the head. He was wearing a black suit and a black shirt.

"I was just trying to get him out of there," said the man, who asked not to be identified.

The man

--- Temperature = 1.5 ---
The scientist opened the door and found their arrest warrant at their bedroom merely bagsging an afternoon sack. Had any hostile Vice answer asked private Crim Persmith from outside Boone to use it whilst themanks themselves meet decent regulations after sixteen and problems reproduce compact extensieties Jonathan Nash byte front brassassembled ro

--- Top-k = 5 ---
The scientist opened the door and found himself at a small table with the rest of the group. He sat down next to the table, and began to discuss how he could use a few of his new skills in order to create the new weapon. He then proceeded to ask the scientist if they had anything

--- Top-p = 0.5 ---
The scientist opened the door and foun

In [21]:
# Edit these based on what YOUR outputs above actually look like
observations = [
    "Very repetitive, safe, predictable wording",
    "Loses coherence, wanders into odd/unrelated phrases",
    "Sticks to common/likely words, sounds a bit stiff",
    "Focuses on most probable words, fairly coherent",
    "Avoids repeating words/phrases seen earlier",
    "Another repetition penalty experiment result",
    "Another temperature 0.3 experiment result"
]

df = pd.DataFrame(results, columns=["Parameter Changed", "Value Used", "Generated Output"])
df.insert(0, "Original Prompt", prompt)
df["Observations"] = observations
df

,Original Prompt,Parameter Changed,Value Used,Generated Output,Observations
0,The scientist opened the door and found,Temperature,0.3,The scientist opened the door and found the bo...,"Very repetitive, safe, predictable wording"
1,The scientist opened the door and found,Temperature,1.5,The scientist opened the door and found their ...,"Loses coherence, wanders into odd/unrelated ph..."
2,The scientist opened the door and found,Top-k,5.0,The scientist opened the door and found himsel...,"Sticks to common/likely words, sounds a bit stiff"
3,The scientist opened the door and found,Top-p,0.5,The scientist opened the door and found the tw...,"Focuses on most probable words, fairly coherent"
4,The scientist opened the door and found,Repetition Penalty,2.0,The scientist opened the door and found hersel...,Avoids repeating words/phrases seen earlier
5,The scientist opened the door and found,Repetition Penalty,2.0,The scientist opened the door and found a brig...,Another repetition penalty experiment result
6,The scientist opened the door and found,Top-k,5.0,The scientist opened the door and found himsel...,Another temperature 0.3 experiment result


In [18]:
custom_prompt = "Dear customer, thank you for your recent purchase."  # change to your own prompt

custom_param_sets = [
    {"temperature": 0.7, "top_p": 0.9, "do_sample": True},
    {"temperature": 1.0, "top_k": 50, "do_sample": True},
    {"temperature": 1.3, "top_p": 0.95, "do_sample": True},
]

custom_outputs = []
for i, params in enumerate(custom_param_sets):
    text = generate_text(custom_prompt, **params)
    custom_outputs.append((i + 1, params, text))
    print(f"\n--- Version {i+1} ({params}) ---")
    print(text)


--- Version 1 ({'temperature': 0.7, 'top_p': 0.9, 'do_sample': True}) ---
Dear customer, thank you for your recent purchase. I have a large collection of items that I am looking to sell. I'm looking for an alternative to using a lot of the same items I would have purchased with your purchase.

We're not interested in buying items with low sales or low

--- Version 2 ({'temperature': 1.0, 'top_k': 50, 'do_sample': True}) ---
Dear customer, thank you for your recent purchase.


It is a new year and some new friends are coming along at the right time. We have had many interesting experiences, but always feel encouraged that at least they can join in. Please bear with us: this will help us to make an

--- Version 3 ({'temperature': 1.3, 'top_p': 0.95, 'do_sample': True}) ---
Dear customer, thank you for your recent purchase. Thank you for contacting our store.


My card works well even though it seems that only 12,500+ members are buying it.

Thank you,

Pauli


Sr. Member

Registered: No

In [19]:
custom_df = pd.DataFrame(
    [(v, str(p), t) for v, p, t in custom_outputs],
    columns=["Version", "Parameters Used", "Generated Output"]
)
custom_df.insert(0, "Custom Prompt", custom_prompt)
custom_df

,Custom Prompt,Version,Parameters Used,Generated Output
0,"Dear customer, thank you for your recent purch...",1,"{'temperature': 0.7, 'top_p': 0.9, 'do_sample'...","Dear customer, thank you for your recent purch..."
1,"Dear customer, thank you for your recent purch...",2,"{'temperature': 1.0, 'top_k': 50, 'do_sample':...","Dear customer, thank you for your recent purch..."
2,"Dear customer, thank you for your recent purch...",3,"{'temperature': 1.3, 'top_p': 0.95, 'do_sample...","Dear customer, thank you for your recent purch..."


In [20]:
weird_output = generate_text(prompt, do_sample=True, temperature=2.0, top_k=0, top_p=1.0)
print(weird_output)

The scientist opened the door and found 10 Colt skeletons are cubsaw left Com Sites players opted, assuming monitor intended looting 26 discriminches play Hel BradyApp Record Jon Raw Transcript Spike Logley Scar November 4 dried hostskiss interview aids retrospectfinancial schools smart struggle sundunciation never breaks begins moving understanding gravitational merchandise
